# 03 — Module 1: anomaly detection

Five detectors (Isolation Forest, LOF, Z-score, IQR; autoencoder built last). Fit on the clean reference batch, score the incoming batch with injected anomalies, evaluate with ROC-AUC and PR-AUC.

### Calibration note (important)
The project's default injection (2% rate, x50 multiplier) was found to be **too weak for this dataset**: amounts are so heavy-tailed (natural 99.9th percentile ~592M, max ~24bn) that a x50 anomaly usually is not an outlier at all. We therefore use a **rare + extreme** injection: rate 0.5%, multiplier 1000, so injected anomalies are genuinely anomalous without clustering. Results are reported with ROC-AUC and PR-AUC (the correct metrics for rare-event detection), not a fixed 0.5 threshold.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from preprocessing import temporal_split, build_module1_features
from injection import inject_anomalies
import module1_anomaly as m1
DATA_PATH = '../data/HI-Small_Trans.csv'   # <-- set to your HPC path

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=['Timestamp','Amount Paid','Is Laundering'])
ref, inc = temporal_split(df, reference_days=3, max_days=10)
# samples keep LOF fast; scale up on the HPC if you wish
ref_fit = ref.sample(100_000, random_state=42).reset_index(drop=True)
inc = inc.sample(100_000, random_state=42).reset_index(drop=True)
print('reference fit:', ref_fit.shape[0], '| incoming test:', inc.shape[0])

## Fit detectors on the clean reference

In [ ]:
Xref, scaler, names = build_module1_features(ref_fit, fit=True)
iso = m1.fit_isolation_forest(Xref)
lof = m1.fit_lof(Xref)
ref_log = Xref[:,0]
print('fitted. features:', names)

## Inject anomalies (rare + extreme) and score the incoming batch

In [ ]:
RATE, MULT = 0.005, 1000   # calibrated for this dataset
inc_c, gt = inject_anomalies(inc, 'Amount Paid', rate=RATE, multiplier=MULT, seed=42)
gt = gt.values
Xinc, _, _ = build_module1_features(inc_c, scaler=scaler, fit=False)
scores = {
    'Isolation Forest': m1.score_isolation_forest(iso, Xinc),
    'LOF':              m1.score_lof(lof, Xinc),
    'Z-score':          m1.zscore_scores(Xinc[:,0], ref_values=ref_log)[0],
    'IQR':              m1.iqr_scores(Xinc[:,0], ref_values=ref_log)[0],
}
rows = []
for name, s in scores.items():
    e = m1.evaluate(s, gt); tk = m1.evaluate_topk(s, gt, k=RATE)
    rows.append([name, e['roc_auc'], e['pr_auc'], tk['precision'], tk['recall']])
pd.DataFrame(rows, columns=['detector','ROC-AUC','PR-AUC','P@topk','R@topk']).round(3)

## Reading the results (honest interpretation)
- **Z-score is the strongest single detector** here because it directly measures amount deviation.
- **Isolation Forest is solid.** Both are global detectors, which suits global amount outliers.
- **LOF fails (ROC-AUC < 0.5).** LOF measures *local* density, but the anomalies are *global* extremes that look uniformly far from everything, so LOF treats them as normal. This is a genuine result, not a bug, and it is direct evidence for the multi-detector design argued in the literature review (Xu et al. 2023; Han et al. 2022).
- **PR-AUC is low across the board** because the natural data already contains huge amounts that compete with the injected anomalies. This is an honest property of financial transaction data and belongs in the limitations.

In [ ]:
plt.figure(figsize=(6,5))
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(gt, s)
    plt.plot(fpr, tpr, label=name)
plt.plot([0,1],[0,1],'k--',alpha=0.4)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('Module 1 — ROC curves'); plt.legend()
plt.tight_layout(); plt.savefig('../results/module1_roc.png', dpi=120); plt.show()

## Next
Autoencoder (built last, optional). Then Module 2 (drift).